# 📦 4. Final Dataset Creation & Integration

Este notebook es la etapa de consolidación final del pipeline de datos de **ProtFlash**. Su objetivo es fusionar la información estructural (secuencias y metadatos) con las representaciones semánticas precomputadas (embeddings) para generar objetos `Dataset` de PyTorch optimizados y listos para el entrenamiento supervisado.

---

## 📋 Descripción General
El script carga por separado los metadatos tabulares (`.csv`) y los tensores numéricos (`.npy`), verifica y garantiza la alineación estricta de índices mediante IDs únicos, y encapsula toda la información en una clase `torch.utils.data.Dataset` personalizada. Esto elimina el overhead de parsing en cada época y asegura compatibilidad directa con `DataLoader` y pipelines de entrenamiento en GPU.

## 🛠️ Funcionalidades Principales

1.  **Carga de Datos Multiformato:**
    * Importa los archivos CSV con identificadores, secuencias crudas y etiquetas de clase.
    * Carga los arrays binarios de NumPy que contienen los tensores de embeddings `(N, 100, 768)`, máscaras de atención `(N, 100)` y labels `(N,)`.
2.  **Verificación y Alineación de Índices:**
    * Compara la columna `id` del CSV con `seq_id` de los embeddings mediante mapeo directo.
    * Aplica reordenamiento programático si se detectan desfases, garantizando integridad total sin pérdida de registros ni fugas de etiquetas.
3.  **Encapsulación en `AMPDataset` (Custom PyTorch Dataset):**
    * Define una clase optimizada que retorna por índice un diccionario estandarizado:
      * `X`: Embedding por residuo `(100, 768)`
      * `M`: Máscara binaria de padding `(100,)`
      * `seq`: Cadena de aminoácidos cruda (para enriquecimiento con AAontology)
      * `y`: Etiqueta de clase `(1,)`
4.  **Serialización Eficiente:**
    * Guarda los datasets como archivos `.pt` nativos de PyTorch, preservando tensores, estructura y metadatos para carga instantánea durante `training.ipynb`.

## 📐 Especificaciones del Output
| 📦 Entregable | 📐 Formato | 📊 Volumen | 🎯 Propósito |
|:---|:---|:---|:---|
| `train_dataset.pt` | `torch.utils.data.Dataset` | ✅ 19,548 muestras | Entrenamiento |
| `val_dataset.pt`   | `torch.utils.data.Dataset` | ✅ 5,125 muestras  | Validación de hiperparámetros |
| `test_dataset.pt`  | `torch.utils.data.Dataset` | ✅ 15,685 muestras | Evaluación final |

* **Dimensión del Embedding:** `(100, 768)` por muestra (ProtFlash per-residue).
* **Total de registros sincronizados:** 40,358 secuencias validadas.

## 📦 Importación de dependencias

In [1]:
import torch
import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset

## ⚙️ Configuración de rutas

In [ ]:
INPUT_EMBEDDINGS = "/kaggle/input/datasets/user/embeddings-dataset"
INPUT_DATA = "/kaggle/input/datasets/user/data-consolidation"
OUTPUT_DATASETS = "/kaggle/working/"

os.makedirs(OUTPUT_DATASETS, exist_ok=True)

## 📂 Carga de metadatos y tensores numéricos

In [3]:
def load_metadata(split):
    df = pd.read_csv(os.path.join(INPUT_DATA, f"{split}.csv"))
    return df[["id", "sequence", "label"]].copy()

train_meta = load_metadata("train")
val_meta   = load_metadata("val")
test_meta  = load_metadata("test")

In [4]:
def load_tensors(split):
    X = np.load(os.path.join(INPUT_EMBEDDINGS, f"{split}_embeddings.npy"))   # (N, 100, 768)
    G = np.load(os.path.join(INPUT_EMBEDDINGS, f"{split}_global.npy"))      # (N, 768)
    M = np.load(os.path.join(INPUT_EMBEDDINGS, f"{split}_masks.npy"))        # (N, 100)
    y = np.load(os.path.join(INPUT_EMBEDDINGS, f"{split}_labels.npy"))       # (N,)
    ids = pd.read_csv(os.path.join(INPUT_EMBEDDINGS, f"{split}_ids.csv"))["seq_id"].tolist()
    return X, G, M, y, ids

train_X, train_G, train_M, train_y, train_ids = load_tensors("train")
val_X, val_G, val_M, val_y, val_ids   = load_tensors("val")
test_X, test_G, test_M, test_y, test_ids  = load_tensors("test")

## 🔍 Verificación y alineación de índices

In [5]:
def verify_alignment(meta_df, tensor_ids, split_name):
    """Compare CSV IDs with tensor IDs to ensure order."""
    meta_ids = meta_df["id"].tolist()
    if meta_ids == tensor_ids:
        print(f"✅ {split_name}: Perfect alignment ({len(meta_ids)} records)")
        return True
    else:
        # Sort tensors by meta_ids
        id_to_idx = {sid: i for i, sid in enumerate(tensor_ids)}
        try:
            order = [id_to_idx[mid] for mid in meta_ids]
            return order
        except KeyError as e:
            raise ValueError(f"❌ ID {e} not found in tensors {split_name}")

print("🔍 Verifying alignment...")
train_order = verify_alignment(train_meta, train_ids, "TRAIN")
val_order   = verify_alignment(val_meta,   val_ids,   "VAL")
test_order  = verify_alignment(test_meta,  test_ids,  "TEST")

# Reorder if necessary
if train_order is not True:
    train_X = train_X[train_order]
    train_G = train_G[train_order]
    train_M = train_M[train_order]
    train_y = train_y[train_order]
if val_order is not True:
    val_X = val_X[val_order]
    val_G = val_G[val_order]
    val_M = val_M[val_order]
    val_y = val_y[val_order]
if test_order is not True:
    test_X = test_X[test_order]
    test_ids
    test_M = test_M[test_order]
    test_y = test_y[test_order]

🔍 Verifying alignment...
✅ TRAIN: Perfect alignment (13826 records)
✅ VAL: Perfect alignment (3614 records)
✅ TEST: Perfect alignment (13762 records)


## 🧩 Clase `AMPDataset` (Custom PyTorch Dataset)

In [6]:
class AMPDataset(Dataset):
    """
    Dataset for training the AMP classifier with Gated Pooling.

    Returns:
        X:     torch.FloatTensor (L_max, 768)    — ProtFlash embeddings
        G:     torch.FloatTensor (768,)          — Global/CLS context embedding
        M:     torch.FloatTensor (L_max,)        — binary mask
        seq:   str                               — AA sequence (for AAontology)
        y:     torch.FloatTensor (1,)            — label
    """
    def __init__(self, X, G, M, sequences, labels):
        self.X = torch.from_numpy(X).float()          # (N, 100, 768)
        self.G = torch.from_numpy(G).float()          # (N, 768)
        self.M = torch.from_numpy(M).float()          # (N, 100)
        self.sequences = sequences                      # list[str]
        self.y = torch.from_numpy(labels).float().unsqueeze(1)  # (N, 1)

        assert len(self.X) == len(self.G) == len(self.M) == len(self.sequences) == len(self.y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return {
            "X": self.X[idx],           # (100, 768)
            "G": self.G[idx],           # (768,)
            "M": self.M[idx],           # (100,)
            "seq": self.sequences[idx], # str
            "y": self.y[idx]            # (1,)
        }

## 💾 Instanciación y serialización persistente

In [7]:
train_dataset = AMPDataset(train_X, train_G, train_M, train_meta["sequence"].tolist(), train_y)
val_dataset   = AMPDataset(val_X,   val_G,   val_M,   val_meta["sequence"].tolist(),   val_y)
test_dataset  = AMPDataset(test_X,  test_G,  test_M,  test_meta["sequence"].tolist(),  test_y)

print(f"📦 Datasets created (con Contexto Global):")
print(f"   Train: {len(train_dataset)} samples")
print(f"   Val:   {len(val_dataset)} samples")
print(f"   Test:  {len(test_dataset)} samples")

📦 Datasets created (con Contexto Global):
   Train: 13826 samples
   Val:   3614 samples
   Test:  13762 samples


In [8]:
torch.save(train_dataset, os.path.join(OUTPUT_DATASETS, "train_dataset.pt"))
torch.save(val_dataset,   os.path.join(OUTPUT_DATASETS, "val_dataset.pt"))
torch.save(test_dataset,   os.path.join(OUTPUT_DATASETS, "test_dataset.pt"))

print(f"💾 Saved in {OUTPUT_DATASETS}:")
print(f"   train_dataset.pt | val_dataset.pt | test_dataset.pt")

💾 Saved in /kaggle/working/:
   train_dataset.pt | val_dataset.pt | test_dataset.pt


## 👁️ Validación de un sample

In [9]:
sample = train_dataset[0]
print(f"\n🔎 Sample [0]:")
print(f"   X shape (Residuos): {sample['X'].shape}  (dtype: {sample['X'].dtype})")
print(f"   G shape (Global):   {sample['G'].shape}  (dtype: {sample['G'].dtype})")
print(f"   M shape:            {sample['M'].shape}  (sum: {sample['M'].sum().item():.0f} posiciones reales)")
print(f"   Seq:                {sample['seq'][:50]}... (len={len(sample['seq'])})")
print(f"   y:                  {sample['y'].item():.0f}")


🔎 Sample [0]:
   X shape (Residuos): torch.Size([100, 768])  (dtype: torch.float32)
   G shape (Global):   torch.Size([768])  (dtype: torch.float32)
   M shape:            torch.Size([100])  (sum: 84 posiciones reales)
   Seq:                FQEGVRLYRICNEKLTEAEEKISVMIEEGNQLVERPFEGGEEFDDLMLLL... (len=84)
   y:                  0


# 🏁 Resumen del Pipeline de Datasets | ProtFlash

Este notebook cierra la fase de **ingesta, vectorización y estructuración**, entregando tres objetos `Dataset` serializados y listos para consumo en el módulo de entrenamiento supervisado.

| 📦 Entregable | 📐 Formato | 📊 Volumen |
|:---|:---|:---|
| `train_dataset.pt` | `torch.utils.data.Dataset` | ✅ 13,826 muestras |
| `val_dataset.pt`   | `torch.utils.data.Dataset` | ✅ 3,614 muestras |
| `test_dataset.pt`  | `torch.utils.data.Dataset` | ✅ 13,762 muestras |

### 📌 Logros Clave
- **Alineación garantizada**: Verificación cruzada de IDs entre CSVs y tensores `.npy` para evitar desajustes label-embedding.
- **Encapsulación PyTorch**: Clase `AMPDataset` optimizada para retornar tensores, máscaras, secuencias crudas y labels en un solo dict.
- **Serialización eficiente**: Guardado en `.pt` para carga instantánea y sin overhead de parsing en `training.ipynb`.